# 02 Symbol and Emotion Extraction

This notebook is self-contained. It extracts candidate symbols with spaCy noun chunks and nouns, filters them with TF-IDF, and detects emotion words with a poetic emotion lexicon.

This is the first notebook that creates the raw material for the knowledge graph. Symbols and emotion words are extracted separately. The important point is that the emotion lexicon only identifies emotional vocabulary; it does not manually decide that a symbol such as moon means grief or hope. Those relations are discovered later by distance inside real poems.


In [1]:
from pathlib import Path
import json
import re
import pandas as pd
import spacy
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
LEXICON_PATH = PROJECT_ROOT / 'data' / 'emotion_lexicon' / 'poetic_emotion_lexicon.json'
POEMS_CLEAN_PATH = PROCESSED_DIR / 'poems_clean.csv'
SYMBOLS_PATH = PROCESSED_DIR / 'extracted_symbols.csv'
EMOTIONS_PATH = PROCESSED_DIR / 'extracted_emotions.csv'
SPACY_MODEL = 'en_core_web_sm'
TFIDF_MAX_DF = 0.7
TFIDF_MIN_DF = 3
TFIDF_MAX_FEATURES = 30000
TOP_TFIDF_TERMS_PER_POEM = 20
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Emotion Lexicon

The emotion list is intentionally broad and poetic. It includes direct emotion words, inflections, and related vocabulary that often carries emotional atmosphere in poems.

A lexicon is used here because it keeps the graph explainable. If the system detects grief, mourning, or tears, we can show exactly which word caused the emotion match. This is different from training a black-box classifier to assign one emotion label to the whole poem.


In [2]:
poetic_emotion_lexicon = {
    'grief': ['grief', 'grieve', 'grieving', 'sorrow', 'sorrowful', 'mourning', 'mourn', 'loss', 'lost', 'tears', 'weeping', 'lament', 'funeral', 'grave', 'bereft', 'absence'],
    'loneliness': ['lonely', 'alone', 'solitude', 'solitary', 'empty', 'emptiness', 'silence', 'silent', 'isolated', 'abandoned', 'forsaken', 'apart', 'unseen'],
    'longing': ['longing', 'yearning', 'yearn', 'desire', 'ache', 'aching', 'waiting', 'distant', 'distance', 'hunger', 'thirst', 'wish', 'wanting', 'reaching'],
    'love': ['love', 'loving', 'beloved', 'heart', 'kiss', 'tender', 'dear', 'passion', 'lover', 'embrace', 'devotion', 'adore', 'cherish'],
    'fear': ['fear', 'afraid', 'terror', 'terrified', 'dread', 'tremble', 'trembling', 'shadow', 'horror', 'panic', 'fright', 'haunted', 'alarm'],
    'hope': ['hope', 'hopeful', 'dawn', 'rise', 'rising', 'light', 'tomorrow', 'bloom', 'blossom', 'promise', 'renew', 'return', 'brighten', 'heal'],
    'joy': ['joy', 'joyful', 'delight', 'glad', 'laughter', 'laugh', 'bright', 'bliss', 'happy', 'happiness', 'cheer', 'singing', 'dance'],
    'anger': ['anger', 'angry', 'rage', 'wrath', 'fury', 'furious', 'bitter', 'bitterness', 'hate', 'hatred', 'resentment', 'scorn'],
    'despair': ['despair', 'hopeless', 'darkness', 'broken', 'ruin', 'ruined', 'nothingness', 'void', 'waste', 'bleak', 'ash', 'collapse'],
    'nostalgia': ['memory', 'memories', 'remember', 'remembered', 'past', 'again', 'childhood', 'old', 'former', 'once', 'home', 'yesterday', 'familiar'],
    'wonder': ['wonder', 'awe', 'mystery', 'strange', 'miracle', 'marvel', 'infinite', 'vast', 'astonished', 'unknown', 'vision', 'dream'],
    'peace': ['peace', 'peaceful', 'still', 'stillness', 'calm', 'quiet', 'rest', 'soft', 'gentle', 'hush', 'sleep', 'serene', 'tranquil'],
    'shame': ['shame', 'ashamed', 'blush', 'hidden', 'hide', 'disgrace', 'humiliation', 'exposed', 'unworthy', 'stain'],
    'guilt': ['guilt', 'guilty', 'blame', 'fault', 'forgive', 'forgiveness', 'sin', 'confess', 'confession', 'burden', 'remorse'],
    'envy': ['envy', 'envious', 'covet', 'want', 'wanting', 'rival', 'green', 'compare', 'possess'],
    'jealousy': ['jealous', 'jealousy', 'guarded', 'possessive', 'betray', 'betrayal', 'suspicion', 'rival', 'faithless'],
    'awe': ['awe', 'reverence', 'reverent', 'sublime', 'vast', 'holy', 'sacred', 'majestic', 'immense', 'humbled'],
    'tenderness': ['tender', 'tenderness', 'gentle', 'soft', 'kind', 'kindness', 'caress', 'cradle', 'warm', 'mercy', 'delicate'],
    'anxiety': ['anxiety', 'anxious', 'worry', 'worried', 'restless', 'uneasy', 'nervous', 'hurry', 'racing', 'uncertain', 'tremor'],
    'melancholy': ['melancholy', 'sad', 'sadness', 'blue', 'dim', 'gray', 'grey', 'heavy', 'weary', 'weariness', 'autumn', 'rain', 'fading'],
    'resilience': ['resilience', 'resilient', 'endure', 'enduring', 'survive', 'survived', 'rise', 'strong', 'strength', 'repair', 'steadfast'],
    'confusion': ['confusion', 'confused', 'lost', 'maze', 'mist', 'blur', 'uncertain', 'question', 'bewildered', 'wandering'],
    'regret': ['regret', 'regretted', 'sorry', 'remorse', 'missed', 'mistake', 'undo', 'would', 'should', 'afterward'],
    'desire': ['desire', 'desired', 'want', 'wanted', 'crave', 'craving', 'hunger', 'hungry', 'thirst', 'thirsty', 'fire', 'yearning'],
    'gratitude': ['gratitude', 'grateful', 'thanks', 'thankful', 'blessing', 'blessed', 'gift', 'given', 'receive', 'grace'],
    'faith': ['faith', 'faithful', 'belief', 'believe', 'prayer', 'pray', 'holy', 'sacred', 'trust', 'church', 'god', 'divine'],
    'doubt': ['doubt', 'doubtful', 'question', 'uncertain', 'maybe', 'perhaps', 'hesitate', 'unbelief', 'unsure'],
    'freedom': ['freedom', 'free', 'release', 'open', 'unbound', 'wing', 'wings', 'flight', 'escape', 'liberty', 'wide', 'breathe'],
    'oppression': ['oppression', 'oppressed', 'chain', 'chains', 'bound', 'captive', 'prison', 'burden', 'weight', 'silenced', 'crushed', 'cage'],
    'mortality': ['death', 'dead', 'dying', 'die', 'mortal', 'mortality', 'grave', 'bone', 'bones', 'dust', 'funeral', 'coffin', 'ghost', 'last'],
    'transcendence': ['transcend', 'transcendent', 'eternal', 'infinite', 'beyond', 'heaven', 'angel', 'soul', 'spirit', 'divine', 'radiance', 'lifted']
}
LEXICON_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(LEXICON_PATH, 'w', encoding='utf-8') as handle:
    json.dump(poetic_emotion_lexicon, handle, ensure_ascii=False, indent=2)

## Extraction Functions

Symbol extraction uses two signals. First, spaCy finds noun chunks and noun/proper-noun tokens because symbols in poetry often appear as concrete nouns or short noun phrases. Second, TF-IDF filters out terms that are too common across the corpus and keeps words that are more distinctive within a poem.

The output keeps token positions and line numbers. These are needed in notebook 03, where each symbol is connected to the nearest emotion word within a short token window.


In [3]:
GENERIC_NOUNS = {'thing', 'things', 'time', 'times', 'way', 'ways', 'day', 'days', 'man', 'men', 'woman', 'women', 'people', 'life', 'world', 'one', 'body', 'part', 'place', 'something', 'nothing', 'everything', 'anything', 'someone', 'everyone', 'word', 'words', 'name', 'names', 'thought', 'thoughts', 'mind', 'heart'}

def line_number_from_char(text, char_index):
    return text[:char_index].count('\n') + 1

def normalize_symbol_text(value):
    text = re.sub(r'\s+', ' ', value.lower()).strip()
    return re.sub(r'^[^a-z]+|[^a-z]+$', '', text)

def is_valid_symbol(text):
    return bool(text and len(text) >= 3 and text not in GENERIC_NOUNS and re.search(r'[a-z]', text))

def extract_candidate_symbols(text, nlp):
    doc = nlp(text)
    rows = []
    seen = set()
    for chunk in doc.noun_chunks:
        tokens = [token for token in chunk if not token.is_stop and token.pos_ not in {'PRON', 'DET'}]
        if tokens:
            symbol_text = normalize_symbol_text(' '.join(token.text for token in tokens))
            lemma = normalize_symbol_text(' '.join(token.lemma_ for token in tokens))
            key = (chunk.start, chunk.end, lemma)
            if is_valid_symbol(lemma) and key not in seen:
                seen.add(key)
                rows.append({'symbol': lemma, 'symbol_text': symbol_text, 'lemma': lemma, 'start_token': chunk.start, 'end_token': chunk.end, 'line_number': line_number_from_char(text, chunk.start_char), 'source_method': 'noun_chunk'})
    for token in doc:
        lemma = normalize_symbol_text(token.lemma_)
        key = (token.i, token.i + 1, lemma)
        if token.pos_ in {'NOUN', 'PROPN'} and not token.is_stop and is_valid_symbol(lemma) and key not in seen:
            seen.add(key)
            rows.append({'symbol': lemma, 'symbol_text': normalize_symbol_text(token.text), 'lemma': lemma, 'start_token': token.i, 'end_token': token.i + 1, 'line_number': line_number_from_char(text, token.idx), 'source_method': 'noun'})
    return rows

def fit_tfidf(poems_df):
    vectorizer = TfidfVectorizer(lowercase=True, stop_words='english', max_df=TFIDF_MAX_DF, min_df=TFIDF_MIN_DF, ngram_range=(1, 2), max_features=TFIDF_MAX_FEATURES)
    matrix = vectorizer.fit_transform(poems_df['clean_text'].fillna(''))
    return vectorizer, matrix

def top_tfidf_terms_for_row(vectorizer, matrix, row_index, top_n=TOP_TFIDF_TERMS_PER_POEM):
    feature_names = vectorizer.get_feature_names_out()
    row = matrix[row_index]
    ranked = sorted(zip(row.indices, row.data), key=lambda item: item[1], reverse=True)[:top_n]
    return {feature_names[index] for index, value in ranked}

def extract_symbols_for_corpus(poems_df, nlp):
    vectorizer, matrix = fit_tfidf(poems_df)
    rows = []
    for index, poem in tqdm(enumerate(poems_df.itertuples(index=False)), total=len(poems_df), desc='Extracting symbols'):
        terms = top_tfidf_terms_for_row(vectorizer, matrix, index)
        candidates = extract_candidate_symbols(poem.poem_text, nlp)
        kept = []
        for item in candidates:
            if item['symbol'] in terms or item['symbol_text'] in terms:
                item = dict(item)
                item['source_method'] = f"{item['source_method']}+tfidf"
                kept.append(item)
        if len(kept) < 3:
            kept = candidates[:10]
        for item in kept:
            item.update({'poem_id': poem.poem_id, 'title': poem.title, 'author': poem.author})
            rows.append(item)
    columns = ['poem_id', 'title', 'author', 'symbol', 'symbol_text', 'lemma', 'start_token', 'end_token', 'line_number', 'source_method']
    return pd.DataFrame(rows, columns=columns)

def emotion_lookup_from_lexicon(lexicon):
    lookup = {}
    for category, words in lexicon.items():
        for word in words:
            lookup.setdefault(word.lower(), set()).add(category)
    return lookup

def extract_emotions_from_text(text, nlp, lexicon):
    lookup = emotion_lookup_from_lexicon(lexicon)
    rows = []
    for token in nlp(text):
        categories = set()
        categories.update(lookup.get(token.text.lower(), set()))
        categories.update(lookup.get(token.lemma_.lower(), set()))
        for category in sorted(categories):
            rows.append({'emotion_category': category, 'matched_word': token.text, 'lemma': token.lemma_.lower(), 'start_token': token.i, 'line_number': line_number_from_char(text, token.idx)})
    return rows

def extract_emotions_for_corpus(poems_df, nlp, lexicon):
    rows = []
    for poem in tqdm(poems_df.itertuples(index=False), total=len(poems_df), desc='Extracting emotions'):
        for item in extract_emotions_from_text(poem.poem_text, nlp, lexicon):
            item.update({'poem_id': poem.poem_id, 'title': poem.title, 'author': poem.author})
            rows.append(item)
    columns = ['poem_id', 'title', 'author', 'emotion_category', 'matched_word', 'lemma', 'start_token', 'line_number']
    return pd.DataFrame(rows, columns=columns)

## Run Extraction

This full-corpus step can take time because spaCy processes every poem. The results are saved to CSV files so later notebooks do not need to repeat the slow extraction step.

If the CSV files already exist and the lexicon or extraction method has not changed, this notebook does not need to be rerun.


In [4]:
poems_df = pd.read_csv(POEMS_CLEAN_PATH)
nlp = spacy.load(SPACY_MODEL)
symbols_df = extract_symbols_for_corpus(poems_df, nlp)
symbols_df.to_csv(SYMBOLS_PATH, index=False)
symbols_df.head()

c:\Users\Lenovo X1 Carbon\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Extracting symbols: 100%|██████████| 13753/13753 [29:28<00:00,  7.78it/s] 


,poem_id,title,author,symbol,symbol_text,lemma,start_token,end_token,line_number,source_method
0,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,window,window,window,13,15,5,noun_chunk+tfidf
1,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,lack,lacks,lack,18,19,5,noun_chunk+tfidf
2,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,suction,suction,suction,20,21,7,noun_chunk+tfidf
3,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,window,window,window,39,41,13,noun_chunk+tfidf
4,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,thatch,thatch,thatch,56,57,17,noun_chunk+tfidf


In [5]:
with open(LEXICON_PATH, 'r', encoding='utf-8') as handle:
    lexicon = json.load(handle)
emotions_df = extract_emotions_for_corpus(poems_df, nlp, lexicon)
emotions_df.to_csv(EMOTIONS_PATH, index=False)
emotions_df.head()

Extracting emotions: 100%|██████████| 13753/13753 [18:44<00:00, 12.23it/s] 


,poem_id,title,author,emotion_category,matched_word,lemma,start_token,line_number
0,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,mortality,bone,bone,1,1
1,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,loneliness,empty,empty,60,17
2,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,freedom,open,open,78,23
3,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,freedom,wing,wing,120,35
4,poem_00001,The New Church,Lucia Cherciu,nostalgia,old,old,1,1
